# Cohort pipeline · all nights

Until now we worked on **one night**. This notebook runs the same steps on **every night you have**, and collects the results in **one table, one row per night**: the table we use for the age analyses on Wednesday and your dashboard on Thursday.

We build it up over the week. Each session adds a block of features to `process_night`:

| Session | Adds |
|---|---|
| **Sat (today)** | loading, artefact detection, **sleep statistics**, data quality |
| Mon | band power per stage (spectral analysis) |
| Tue | automated staging (YASA) vs. the experts |
| Wed | spindles and slow waves |

**How to work:** run from top to bottom. Cells marked 👉 have a parameter to change and rerun. Running all 153 nights takes about 5–15 min.

In [ ]:
%matplotlib inline
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import yasa
from scipy.stats import spearmanr

sys.path.insert(0, str(Path("..").resolve()))  # so Python finds sleep_utils.py and cohort.py in the repo folder
from sleep_utils import SUBJECTS_CSV, bad_epochs_table, load_subject, make_epochs, night_code
from cohort import sleep_architecture_features

mne.set_log_level("ERROR")

RESULTS_DIR = Path("..") / "results"
COHORT_FILE = RESULTS_DIR / "cohort_features.csv"

## 1. The nights, and which ones you include

All your choices about which nights to use live in **`database.csv`** in the repo folder: one row per night (the columns of `SC-subjects.csv`, plus the night `code`), with two extra columns:
- **`include`**: `True` or `False`. The pipeline only runs the nights with `True`, and the analyses only use those.
- **`exclude_reason`**: why you left the night out. Always write one: you'll have to defend it in your presentation.

The first run creates `database.csv` with all nights included. After that, it is **never overwritten**: this notebook will get new sections during the week, and your choices stay in your file. It's in `.gitignore`, so `git pull` doesn't touch it either.

👉 To leave out a night, open `database.csv` in VS Code, set `include` to `False` and write the reason, save, and rerun from here. To leave out a person, do it for all their nights. You don't need to rerun the pipeline (Section 3): Section 4 only uses the included nights. (If you edit it in Excel, save it as CSV again.)

In [ ]:
DATABASE_FILE = Path("..") / "database.csv"
if not DATABASE_FILE.exists():  # first run: SC-subjects.csv + code, include and exclude_reason, all nights included
    subjects = pd.read_csv(SUBJECTS_CSV)
    subjects.insert(0, "code", [night_code(s, n) for s, n in zip(subjects["subject"], subjects["night"])])
    subjects["include"] = True
    subjects["exclude_reason"] = ""
    subjects.to_csv(DATABASE_FILE, index=False)
    print(f"Created {DATABASE_FILE.resolve()}")

subjects = pd.read_csv(DATABASE_FILE, dtype={"exclude_reason": str})
subjects["exclude_reason"] = subjects["exclude_reason"].fillna("")

# include must be True or False (also accepts true/false/1/0, e.g. after editing in Excel)
include = subjects["include"].astype(str).str.strip().str.lower().map({"true": True, "1": True, "false": False, "0": False})
if include.isna().any():
    raise ValueError(f"include must be True or False. Check these nights in {DATABASE_FILE.name}: "
                     f"{list(subjects.loc[include.isna(), 'code'])}")
subjects["include"] = include.astype(bool)

no_reason = subjects.loc[~subjects["include"] & (subjects["exclude_reason"].str.strip() == ""), "code"]
if len(no_reason):
    print(f"⚠️ Excluded without a reason: {list(no_reason)}")

nights = subjects[subjects["include"]].reset_index(drop=True)
print(f"{len(nights)} of {len(subjects)} nights included ({nights['subject'].nunique()} people)")
subjects.loc[~subjects["include"], ["code", "age", "exclude_reason"]]


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(subjects["age"], bins=range(20, 105, 5), color="lightgray", label="all nights")
ax.hist(nights["age"], bins=range(20, 105, 5), color="tab:blue", label="included")
ax.set(xlabel="Age (years)", ylabel="Nights")
ax.legend()
plt.show()

**Question:** Look at the age histogram. Which ages are missing? What does that mean for "the effect of age"?

## 2. One function per night

`process_night` does, for one night, what you did step by step in Sessions 2–4, and returns a **dict of features** (one row of the cohort table).

- **Loading:** `load_subject` (lights-off → end of the sleep period: naps the next day are left out, see Session 4), filter 0.1–40 Hz, 30-s epochs
- **Artefacts:** the course detector + movement time (`bad_epochs_table`, Session 3). The table of bad epochs is saved in `results/<night>/bad_epochs.csv` (not `bad_epochs_auto.csv`, so this morning's file with all methods stays)
- **Sleep statistics:** `sleep_architecture_features` from `cohort.py`: YASA on the **unedited** hypnogram (Session 4), plus the REM latency from sleep onset, the stage shifts per hour, the **sleep continuity index** and the **transition rate** out of each stage (Session 4, Exercises 4.2 and 5.2). Open `cohort.py` to see the code
- **Data quality:** % bad epochs, overall and per stage, and how many clean epochs are left per stage (these are what the EEG features of the next sessions are computed on)

Read the function once before you run it.

In [ ]:
def process_night(subject, night):
    """All features of one night, as a dict."""
    code = night_code(subject, night)
    row = subjects[(subjects["subject"] == subject) & (subjects["night"] == night)].iloc[0]
    features = {"code": code, "subject": subject, "night": night, "age": row["age"],
                "sex": "F" if row["sex_F"] == 1 else "M"}

    # --- Loading
    raw = load_subject(subject, night)
    raw.filter(l_freq=0.1, h_freq=40, picks=["eeg", "eog"])
    epochs, stages = make_epochs(raw)

    # --- Artefacts (Session 3): only for the EEG features, never for the sleep statistics
    bad = bad_epochs_table(epochs, stages)
    (RESULTS_DIR / code).mkdir(parents=True, exist_ok=True)
    bad.to_csv(RESULTS_DIR / code / "bad_epochs.csv", index=False)

    # --- Sleep statistics (Session 4): the unedited expert hypnogram
    features.update(sleep_architecture_features(stages))

    # --- Data quality
    features["pct_bad"] = 100 * bad["bad"].mean()
    for stage in ["WAKE", "N2", "N3", "REM"]:
        in_stage = bad["stage"] == stage
        features[f"pct_bad_{stage}"] = 100 * bad.loc[in_stage, "bad"].mean() if in_stage.any() else np.nan
        features[f"n_clean_{stage}"] = (in_stage & ~bad["bad"]).sum()

    # --- Monday: band power per stage, on the clean epochs only (bad["bad"] == False)

    # --- Tuesday: automated staging (YASA) vs. the experts

    # --- Wednesday: spindles and slow waves, on the clean epochs only

    return features

### Test it on one night first

Always test on a night you know. For SC4002 you should find the numbers of this morning (53 bad epochs, 5.2 %) and this afternoon (TST 472 min, SE 92.3 %).

In [ ]:
t0 = time.time()
test = process_night(0, 2)
print(f"{time.time() - t0:.1f} s")
pd.Series(test)

## 3. Run all nights

👉 First run with `N_NIGHTS = 5` to check that everything works and how long one night takes. Then set `N_NIGHTS = None` (all included nights) and rerun: a few minutes.

If a night fails, the loop doesn't stop: it notes the error and goes on. Look at the failures at the end.

In [ ]:
N_NIGHTS = 5  # 👉 None = all nights

todo = nights if N_NIGHTS is None else nights.head(N_NIGHTS)
rows, failed = [], {}
t0 = time.time()
for k, (subject, night) in enumerate(zip(todo["subject"], todo["night"]), start=1):
    try:
        rows.append(process_night(subject, night))
    except Exception as err:  # keep going, but remember what failed
        failed[night_code(subject, night)] = repr(err)
    print(f"\r{k}/{len(todo)} nights, {time.time() - t0:.0f} s", end="")

cohort = pd.DataFrame(rows)
cohort.to_csv(COHORT_FILE, index=False)
print(f"\nSaved {len(cohort)} nights to {COHORT_FILE}. Failed: {len(failed)}")
failed

## 4. Explore the cohort

From here on, nothing is recomputed: we read the saved table and keep the included nights. After a restart, run the setup cell and Section 1, then start here.

In [ ]:
cohort = pd.read_csv(COHORT_FILE)
cohort = cohort[cohort["code"].isin(nights["code"])]  # only the nights included in Section 1
print(f"{len(cohort)} nights, {cohort['subject'].nunique()} subjects")
cohort.describe().T.round(1)

### One statistic vs. age

👉 Change `METRIC` and rerun: try `SE`, `TST`, `WASO`, `%N3`, `%REM`, `N3`, `SOL`, `Lat_REM_onset`, `shifts_per_h`, `SFI`, `SCI`, `rate_N1`, `rate_N3`, `rate_REM`. ρ is Spearman's rank correlation with age.

In [ ]:
METRIC = "%N3"  # 👉 change me

fig, ax = plt.subplots(figsize=(6, 4))
for sex, color in [("F", "tab:red"), ("M", "tab:blue")]:
    d = cohort[cohort["sex"] == sex]
    ax.scatter(d["age"], d[METRIC], s=18, alpha=0.7, color=color, label=sex)
rho, p = spearmanr(cohort["age"], cohort[METRIC], nan_policy="omit")
ax.set(xlabel="Age (years)", ylabel=METRIC, title=f"{METRIC} vs. age: ρ = {rho:.2f} (p = {p:.1g}, n = {len(cohort)} nights)")
ax.legend(title="Sex")
plt.show()

### All statistics at once

In [ ]:
METRICS = ["TST", "SE", "SOL", "WASO", "Lat_REM_onset", "%N1", "%N2", "%N3", "%REM", "SFI",
           "shifts_per_h", "SCI", "rate_N1", "rate_N2", "rate_REM"]

fig, axes = plt.subplots(3, 5, figsize=(16, 9), sharex=True)
for ax, m in zip(axes.flat, METRICS):
    ax.scatter(cohort["age"], cohort[m], s=8, alpha=0.6)
    rho, _ = spearmanr(cohort["age"], cohort[m], nan_policy="omit")
    ax.set_title(f"{m} (ρ = {rho:.2f})")
for ax in axes[-1]:
    ax.set_xlabel("Age (years)")
plt.tight_layout()
plt.show()

**Questions**
1. Which statistics change most with age? Does that match what you know (T2)?
2. `SCI`, the transition rates and `shifts_per_h` all describe fragmentation. Do they carry the same information? Plot two of them against each other.
3. Most subjects have **two nights**. Each dot is a night, not a person. Why is that a problem for the p-value above? (Wednesday's age prediction must handle this too.)
4. Which statistics depend on our choice of the end of the night (the sleep period ends at the first wake bout of ≥ 3 h, see `load_subject`)? Would their age trend survive another choice?

### Night 1 vs. night 2

People often sleep worse in the first night with the equipment (the **first-night effect**). Is that the case here?

👉 Change `METRIC` and rerun.

In [ ]:
METRIC = "SE"  # 👉 change me

paired = cohort.pivot_table(index="subject", columns="night", values=METRIC).dropna()
fig, ax = plt.subplots(figsize=(4, 4))
ax.plot([1, 2], paired[[1, 2]].T, color="gray", alpha=0.4, lw=0.8)
ax.plot([1, 2], paired[[1, 2]].median(), color="black", marker="o", lw=2, label="median")
ax.set(xticks=[1, 2], xticklabels=["night 1", "night 2"], ylabel=METRIC, title=f"{len(paired)} subjects")
ax.legend()
plt.show()
print(f"Night 2 - night 1: median {(paired[2] - paired[1]).median():.1f}")

### Data quality across the cohort

How many epochs does the artefact detector reject, per night? Are some nights much worse, and does that depend on age?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(cohort["pct_bad"], bins=30)
axes[0].set(xlabel="% bad epochs in the night", ylabel="Nights")
axes[1].scatter(cohort["age"], cohort["pct_bad"], s=12, alpha=0.6)
axes[1].set(xlabel="Age (years)", ylabel="% bad epochs")
plt.show()

cohort.nlargest(5, "pct_bad")[["code", "age", "pct_bad", "pct_bad_WAKE", "pct_bad_N2", "pct_bad_N3", "pct_bad_REM",
                               "n_clean_N2", "n_clean_N3", "n_clean_REM"]].round(1)

**Questions**
1. Which stage loses most epochs? Why is that less of a problem for the EEG features than losing N2/N3 epochs?
2. Some nights have very few clean N3 epochs (`n_clean_N3`). What will that mean for "N3 delta power" on Monday, especially in old subjects?
3. Would you exclude whole nights? Write down a rule (e.g. "fewer than … clean N2 epochs") and count how many nights it removes, per age group. Does your rule remove the old subjects selectively? If you keep it, set `include` to `False` for these nights in `database.csv` (Section 1), with the rule as the reason.

In [ ]:
# Your exclusion rule (question 3)

## What's next

On Monday you add **band power per stage** to `process_night` (on the clean epochs only) and rerun the cohort. Keep this notebook: it grows with you, and its table feeds Wednesday's age prediction and Thursday's dashboard.